In [1]:
import math
import pandas as pd

import numpy as np
from scipy.stats import spearmanr
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [2]:
df = pd.read_parquet("train_team_track.parquet", engine='fastparquet')

In [ ]:
df.info()

In [ ]:
df.isna().sum()

Нет ни нулевых значений, ни категориальных данных

In [ ]:
df.head()

In [ ]:
variable_features = df.columns[3:]
static_features = df.columns[:2]

In [ ]:
print(f"Количество уникальных route_id: {df['route_id'].nunique()}")
print(f"Количество уникальных office_from_id: {df['office_from_id'].nunique()}")
print(f"Количество уникальных пар (route_id, office_from_id): {df[['route_id', 'office_from_id']].drop_duplicates().shape[0]}")

In [ ]:
routes_with_offices = df.groupby('office_from_id')['route_id'].nunique()
routes_with_offices[routes_with_offices != 1]

Каждому складу принадлежит несколько маршрутов, но каждому маршруту принадлежит один склад

In [ ]:
amount_rows_in_routes_with_office = df.groupby(['office_from_id', 'route_id']).size()
amount_rows_in_routes_with_office[amount_rows_in_routes_with_office != 4342]

Все группы склад-маршрут имеют одинаковый размер

In [ ]:
timestamps = df.groupby(['office_from_id', 'route_id'])['timestamp'].agg(['first', 'last'])
timestamps[(timestamps['first']!=timestamps.iloc[0]['first']) | (timestamps['last']!=timestamps.iloc[0]['last'])]

In [ ]:
print(f"Начальное время отследивания: {timestamps.iloc[0]['first']}")
print(f"Конечное время отследивания: {timestamps.iloc[0]['last']}")

Во всех группах склад-маршрут время началное и конечное время одинаковы

Можно отметить, что даты приходятся на праздники: 8 марта, 1 апреля, 1 мая, 9 мая

In [ ]:
df['day'] = pd.to_datetime(df['timestamp']).dt.dayofweek

In [ ]:
status_cols = ['status_1', 'status_2', 'status_3', 'status_4', 
               'status_5', 'status_6', 'status_7', 'status_8']

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

for i, status in enumerate(status_cols):
    sns.boxplot(data=df, x='day', y=status, ax=axes[i])
    axes[i].set_title(status)

plt.tight_layout()
plt.show()

In [ ]:
pd.set_option('display.max_columns', None)
descr = df.groupby('day')[variable_features[:-1]].describe()
descr.index = descr.index.map({0:'Mon', 1:'Tue', 2:'Wed', 3:'Thu', 4:'Fri', 5:'Sat', 6:'Sun'})
print(descr)
pd.set_option('display.max_columns',20)

Сильный разброс, медиана во всех статусах в разы меньше среднего. Но можно отметить, что со вторника по среду наблюдается прирост по всем статусам, а также прирост по таргету. 

In [ ]:
df.iloc[1599299:1599310]

In [ ]:
for i in range(8):
    print(df.iloc[1599299+i:1599299+i+4][variable_features[:-2]].sum().sum())

Видно, что несмотря на сильную убыль количества товара суммарно по всем статусам, значение таргета остаётся неизменным (с 3:00 по 4:00). Либо не все товары, прошедшие конкретные этапы обработки, по итогу оказываются выгруженными. Либо емкость, в которой измеряется target_2h, способна вместить довольно много товаров

In [ ]:
corr = df[['status_1','status_2','status_3','status_4',
           'status_5','status_6','status_7','status_8','target_2h']].corr(method='spearman')
print(corr['target_2h'].sort_values(ascending=False))

Возможно, корреляция не совсем корректна из-за того, что мы учитываем количество товаров, а не разницу между данными, переставшими отслеживаться (из-за того, что "двухчасовое окно", которое отслеживает таргет, сдвинулось), и новыми данными, которые только попали в "двухчасовое окно"

Думаю, стоит посмотреть на общее распределение значений, чтобы увидеть общую картину, но будет отсматривать на десятитысячной выборке, при этом случайно выбирая экземпляры. да, не учитываем время, но для текущей цели посмотреть общее распределение это неважно

In [ ]:
df_sample, _ = train_test_split(
    df, 
    train_size=10000,  
    stratify=df[static_features], 
    random_state=1
)

In [ ]:
def density_hist(feature, df):
    plt.hist(df[feature], bins=50, density=True)
    plt.xlabel(feature)
    plt.ylabel('density')
    plt.show()

In [ ]:
for feature in df_sample.columns[3:-1]:
    density_hist(feature, df_sample)

Никаких артефактов не видно, данные вполне логичны

In [ ]:
corr_df_sample = df_sample[variable_features].corr(method='spearman')

In [ ]:
plt.figure(figsize=(12,10))
ax = sns.heatmap(data=corr_df_sample, annot=True, cmap='Oranges', fmt='.2f')
ax.set_xticklabels(ax.get_xticklabels(), fontsize=10)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=10)
plt.title('truncated dataset')

plt.tight_layout()
plt.show()

In [17]:
df['hours'] = pd.to_datetime(df['timestamp']).dt.hour + pd.to_datetime(df['timestamp']).dt.minute/60
df['hours']

0           0.0
1           0.5
2           1.0
3           1.5
4           2.0
           ... 
4341995     8.5
4341996     9.0
4341997     9.5
4341998    10.0
4341999    10.5
Name: hours, Length: 4342000, dtype: float64

In [18]:
def density_dayweek_time():
    df_grouped_dayweek_time = df.groupby(['day', 'hours'])[variable_features[:-1]].mean().reset_index()
    
    _, axes = plt.subplots(9, 1, figsize=(15, 100))
    axes = axes.flatten()

    day_names = {0:'Mon',1:'Tue',2:'Wed',3:'Thu',4:'Fri',5:'Sat',6:'Sun'}

    for i, feature in enumerate(variable_features[:-1]):
        for day in range(7):
            data = df_grouped_dayweek_time[df_grouped_dayweek_time['day'] == day]
            axes[i].plot(data['hours'], data[feature], label=day_names[day])

        axes[i].set_title(feature, fontsize=20)
        axes[i].set_xlabel('Hour', fontsize=16)
        axes[i].set_ylabel('Mean value', fontsize=16)
        axes[i].set_xticks(np.arange(0, 24, 0.5))
        axes[i].set_xticklabels([f'{int(h)}:{int((h%1)*60):02d}' for h in np.arange(0, 24, 0.5)], rotation=45)
        axes[i].legend(fontsize=16)

    plt.tight_layout()
    plt.show()

In [ ]:
density_dayweek_time()

Как видно из графиков, в средних значениях статусов наблюдаются определённые тенденции:
- спад status_1 наблюдается с 0:00 до 5:00, рост с 5:00 до 14:00, потом незначительные колебания
- спад status_2 наблюдается с 0:00 до 4:30, рост с 4:30 до 12:00, потом незначительные колебания
- спад status_3 наблюдается с 0:00 до 7:30, рост с 7:30 до 14:00, потом спад с 14:00 до 0:00 следующего дня
- спад status_4 наблюдается с 0:00 до 8:00, рост с 8:00 до 17:00, потом спад с 17:00 до 21:30, потом рост до 0:00 следующего дня
- спад status_5 наблюдается с 0:00 до 8:00, рост с 8:00 до 16:00, потом спад с 16:00 до 21:00, потом рост до 0:00 следующего дня
- спад status_6 наблюдается с 0:00 до 9:30, рост с 9:40 до 16:30, потом спад с 16:30 до 21:30, потом рост до 0:00 следующего дня
- status_7 и status_8 проявляют себя очень нестабильно, постоянно скачут, но всё равно общий тренд отследить возможно
- у status_7 спад наблюдается с 0:00 до 10:30, причём с 8:30 до 10:30 очень резкий скачок вниз, там значение за 1.5 часа уменьшается на 1000, с 10:30 до 20:30 рост, а потом с 20:30 до 21:30 снова резкий скачок вниз, за 1 час уменьшается на 1000, дальше до 0:00 рост
- у status_8 очень большие колебания, но всё равно можно отследить некий тренд, с 0:00 до 10:30 спад, потом до следующего дня рост
- у target_2h наблюдается спад с 0:00 до 10:30, причём с 8:30 до 10:30 спад более резкий, чем до этого, рост с 10:30 до 18:30, потом снова спад с 18:30 до 23:00, дальше рост до 0:00 следующего дня

### Временные интервалы ключевых изменений
| Статус | Начало спада | Конец спада | Начало роста | Пик активности | Вечерний спад |
|--------|--------------|-------------|--------------|----------------|---------------|
| status_1 | 00:00 | 05:00 | 05:00 | 14:00 | - |
| status_2 | 00:00 | 04:30 | 04:30 | 12:00 | - |
| status_3 | 00:00 | 07:30 | 07:30 | 14:00 | с 14:00 |
| status_4 | 00:00 | 08:00 | 08:00 | 17:00 | 17:00-21:30 |
| status_5 | 00:00 | 08:00 | 08:00 | 16:00 | 16:00-21:00 |
| status_6 | 00:00 | 09:30 | 09:40 | 16:30 | 16:30-21:30 |
| status_7 | 00:00 | 10:30 | 10:30 | 20:30 | 20:30-21:30 |
| status_8 | 00:00 | 10:30 | 10:30 | 00:00 | - |
| target_2h | 00:00 | 10:30 | 10:30 | 18:30 | 18:30-23:00 |

### Сдвиг пиков активности
| Этап | Статусы | Пик активности | Относительно предыдущего |
|------|---------|----------------|--------------------------|
| Поступление | 1-3 | 12:00-14:00 | базовый |
| Обработка | 4-6 | 16:00-17:00 | +2-4 часа |
| Готовность | 7-8 | 20:30 | +4-6 часов |
| Отгрузка | target_2h | 18:30 | +2-4 часа |

In [ ]:
rolling_window_sum_status = pd.DataFrame()
rolling_window_sum_status

In [ ]:
for feature in variable_features[:-1]:
    rolling_window_sum_status[f'status_{feature}_sum'] = df[feature].rolling(window=4).sum()

rolling_window_sum_status['target_2h'] = df['target_2h']
rolling_window_sum_status

In [ ]:
corr = rolling_window_sum_status.iloc[3:].corr(method='spearman')
corr

In [ ]:
plt.figure(figsize=(12,10))
sns.heatmap(corr, cmap='Oranges', annot=True)
plt.show()

In [ ]:
for i in range(1, 8):
        for j in range(i+1, 9):
            lag_corr = df[f'status_{i}'].corr(df[f'status_{j}'].shift(-1))
            print(f"status_{i}(t) → status_{j}(t+1): {lag_corr:.3f}")

# Rough solution without trainable models

In [20]:
df_temp = df.drop('timestamp', axis=1, inplace=False)

In [ ]:
df_train = pd.DataFrame(columns=df_temp.columns)
df_test = pd.DataFrame(columns=df_temp.columns)

first_test_index = round(4342*0.8)

for group, data in df_temp.groupby(["office_from_id", "route_id"]):
    df_train = pd.concat([df_train, data.reset_index(drop=True).iloc[:first_test_index]], ignore_index=True)
    df_test = pd.concat([df_test, data.reset_index(drop=True).iloc[first_test_index:]], ignore_index=True)
df_train

In [30]:
df_train['hours'] = pd.to_datetime(df_train['timestamp']).dt.hour + pd.to_datetime(df_train['timestamp']).dt.minute/60
df_test['hours'] = pd.to_datetime(df_test['timestamp']).dt.hour + pd.to_datetime(df_test['timestamp']).dt.minute/60

In [23]:
df_train_temp = df_train.drop(['office_from_id', 'route_id'], axis=1, inplace=False)
df_train_temp

,Unnamed: 0,timestamp,status_1,status_2,status_3,status_4,status_5,status_6,status_7,status_8,target_2h,hour_sin,hour_cos,dow_sin,dow_cos,hours
0,0,2025-03-01 00:00:00,3.131336,2.466340,3.060036,3.145953,2.801811,2.174074,0.522429,-0.319262,134.0,0.000000e+00,1.000000,-0.974928,-0.222521,0.0
1,1,2025-03-01 00:30:00,2.802500,2.161388,2.369057,2.925726,2.952444,3.106140,4.752760,0.055305,128.0,2.083183e-02,0.999783,-0.974928,-0.222521,0.5
2,2,2025-03-01 01:00:00,2.428860,1.831844,2.901146,2.663633,2.874169,3.362193,-0.254783,1.745945,127.0,-2.449294e-16,1.000000,-0.974928,-0.222521,1.0
3,3,2025-03-01 01:30:00,1.896002,1.280965,2.686405,2.526565,3.093520,2.564797,-0.454539,1.743266,129.0,2.083183e-02,0.999783,-0.974928,-0.222521,1.5
4,4,2025-03-01 02:00:00,1.583568,0.916991,2.678179,2.724998,3.009329,3.407821,3.775589,-0.269426,124.0,-4.898587e-16,1.000000,-0.974928,-0.222521,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3907995,3907995,2025-05-21 07:30:00,-0.522262,-0.356918,-0.551155,-0.254375,-0.139396,-0.303611,-0.267421,0.597060,80.0,2.083183e-02,0.999783,0.974928,-0.222521,7.5
3907996,3907996,2025-05-21 08:00:00,-0.522262,-0.160176,-0.551155,-0.349004,-0.072499,-0.221213,-0.454539,0.443804,67.0,-1.959435e-15,1.000000,0.974928,-0.222521,8.0
3907997,3907997,2025-05-21 08:30:00,-0.522262,-0.091316,-0.551155,-0.206774,-0.121193,0.195204,0.336942,0.347885,65.0,2.083183e-02,0.999783,0.974928,-0.222521,8.5
3907998,3907998,2025-05-21 09:00:00,-0.522262,-0.160176,-0.551155,-0.447074,-0.404710,-0.101161,-0.159593,1.020390,59.0,-2.204364e-15,1.000000,0.974928,-0.222521,9.0


In [26]:
df_train_temp.drop('timestamp', axis=1, inplace=True)

In [27]:
df_mean_value = df_train_temp.groupby('hours').mean()
df_mean_value

,Unnamed: 0,status_1,status_2,status_3,status_4,status_5,status_6,status_7,status_8,target_2h,hour_sin,hour_cos,dow_sin,dow_cos
hours,,,,,,,,,,,,,,
0.0,1953990.0,0.076555,0.063034,0.094455,0.080923,0.084847,0.076279,0.045634,0.022595,70.429585,0.000000e+00,1.000000,-1.353931e-21,0.021975
0.5,1953991.0,-0.011475,-0.055326,0.066225,0.052281,0.066112,0.061491,0.029930,0.017741,70.600963,2.083183e-02,0.999783,-1.353931e-21,0.021975
1.0,1953992.0,-0.102424,-0.182088,0.050461,0.041673,0.055404,0.042983,0.030774,0.017766,70.504098,-2.449294e-16,1.000000,-1.353931e-21,0.021975
1.5,1953993.0,-0.075548,-0.311428,0.001735,0.023121,0.033504,0.054105,0.011778,0.025914,69.954159,2.083183e-02,0.999783,-1.353931e-21,0.021975
2.0,1953994.0,-0.256655,-0.413104,-0.000048,0.007625,0.029235,0.029278,0.011826,0.014538,69.421402,-4.898587e-16,1.000000,-1.353931e-21,0.021975
2.5,1953995.0,-0.321401,-0.487706,-0.052402,-0.014888,0.018348,0.019577,0.010516,0.016570,68.754549,2.083183e-02,0.999783,-1.353931e-21,0.021975
3.0,1953996.0,-0.366748,-0.536576,-0.095166,-0.035596,0.005263,0.009676,0.006039,-0.018162,68.316488,-7.347881e-16,1.000000,-1.353931e-21,0.021975
3.5,1953997.0,-0.396048,-0.568130,-0.157326,-0.065151,-0.012478,0.008803,-0.001696,-0.014852,67.844390,2.083183e-02,0.999783,-1.353931e-21,0.021975
4.0,1953998.0,-0.411491,-0.580618,-0.198969,-0.097595,-0.031350,-0.029325,0.010519,0.005009,66.961378,-9.797174e-16,1.000000,-1.353931e-21,0.021975


In [29]:
df_test_temp = df_test.drop(['office_from_id', 'route_id'], axis=1, inplace=False)
df_test_temp

,Unnamed: 0,timestamp,status_1,status_2,status_3,status_4,status_5,status_6,status_7,status_8,target_2h,hour_sin,hour_cos,dow_sin,dow_cos
0,0,2025-05-21 10:00:00,2.566875,2.131877,1.938710,2.681412,2.037725,1.256626,0.452311,-0.436079,103.0,-2.449294e-15,1.000000,0.974928,-0.222521
1,1,2025-05-21 10:30:00,2.893710,2.712268,2.363861,2.937196,2.226130,2.350387,3.509996,-0.486986,101.0,2.083183e-02,0.999783,0.974928,-0.222521
2,2,2025-05-21 11:00:00,3.117735,3.125428,2.853955,3.140792,2.170155,2.300328,-0.383809,0.182304,99.0,-9.799650e-15,1.000000,0.974928,-0.222521
3,3,2025-05-21 11:30:00,3.449371,2.943441,3.288198,2.921138,2.724447,1.969852,-0.454539,1.605014,112.0,2.083183e-02,0.999783,0.974928,-0.222521
4,4,2025-05-21 12:00:00,3.414167,2.623734,3.908608,3.227965,2.655274,3.814050,3.162462,-0.239954,103.0,-2.939152e-15,1.000000,0.974928,-0.222521
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
433995,433995,2025-05-30 08:30:00,-0.522262,-0.160176,-0.551155,-0.287638,-0.409716,-0.603520,-0.454539,0.075132,47.0,2.083183e-02,0.999783,-0.433884,-0.900969
433996,433996,2025-05-30 09:00:00,-0.522262,-0.017537,-0.551155,-0.378253,-0.380590,-0.026294,-0.352622,0.139971,39.0,-2.204364e-15,1.000000,-0.433884,-0.900969
433997,433997,2025-05-30 09:30:00,-0.522262,0.036567,-0.551155,-0.394884,-0.208569,-0.432080,-0.454539,0.010829,35.0,2.083183e-02,0.999783,-0.433884,-0.900969
433998,433998,2025-05-30 10:00:00,-0.522262,-0.076560,-0.551155,-0.388002,0.087236,0.540299,-0.454539,0.772287,40.0,-2.449294e-15,1.000000,-0.433884,-0.900969


In [31]:
df_temp = df_mean_value.loc[0.0:df_test.iloc[0]['hours']-0.5]
df_mean_value.drop(df_mean_value.loc[0.0:df_test.iloc[0]['hours'] - 0.5].index, inplace=True)
df_mean_value = pd.concat([df_mean_value, df_temp])
df_mean_value

,Unnamed: 0,status_1,status_2,status_3,status_4,status_5,status_6,status_7,status_8,target_2h,hour_sin,hour_cos,dow_sin,dow_cos
hours,,,,,,,,,,,,,,
10.0,1953986.0,0.028950,0.080053,-0.115968,-0.175979,-0.162894,-0.189824,-0.183605,-0.065961,56.795420,-2.449294e-15,1.000000,-1.203615e-02,0.024993
10.5,1953987.0,0.099730,0.160888,-0.056620,-0.107224,-0.102212,-0.144209,-0.111272,-0.061589,55.543889,2.083183e-02,0.999783,-1.203615e-02,0.024993
11.0,1953988.0,0.139315,0.222990,0.034206,-0.047842,-0.054766,-0.091555,-0.073619,-0.070867,57.067185,-9.799650e-15,1.000000,-1.203615e-02,0.024993
11.5,1953989.0,0.174408,0.274305,0.080942,0.006493,-0.011330,-0.027491,-0.053343,-0.047543,62.209395,2.083183e-02,0.999783,-1.203615e-02,0.024993
12.0,1953990.0,0.202848,0.312351,0.150501,0.054711,0.018790,-0.000740,-0.027370,-0.048842,64.349420,-2.939152e-15,1.000000,-1.203615e-02,0.024993
12.5,1953991.0,0.196787,0.309231,0.162134,0.093614,0.042568,0.023263,-0.021948,-0.045238,66.107864,2.083183e-02,0.999783,-1.203615e-02,0.024993
13.0,1953992.0,0.226631,0.336245,0.196355,0.122742,0.061695,0.043746,-0.003634,-0.012286,67.446383,3.921346e-15,1.000000,-1.203615e-02,0.024993
13.5,1953993.0,0.229879,0.311866,0.195688,0.139742,0.075604,0.069934,-0.005005,-0.025380,68.368259,2.083183e-02,0.999783,-1.203615e-02,0.024993
14.0,1953994.0,0.212778,0.309719,0.220983,0.152930,0.089740,0.073720,0.015521,-0.007857,69.325901,-3.429011e-15,1.000000,-1.203615e-02,0.024993


In [32]:
df_mean_value = df_mean_value.reset_index(drop=True)
n = ((df_test['office_from_id']!=1)|(df_test['route_id']!=33)).idxmax()

df_mean_expanded = df_mean_value.iloc[
    np.tile(np.arange(len(df_mean_value)), n // len(df_mean_value) + 1)
][:n].reset_index(drop=True)

n = df_test_temp.shape[0]

df_mean_expanded = df_mean_expanded.iloc[
    np.tile(np.arange(len(df_mean_expanded)), n // len(df_mean_expanded) + 1)
][:n].reset_index(drop=True)

df_mean_expanded

,Unnamed: 0,status_1,status_2,status_3,status_4,status_5,status_6,status_7,status_8,target_2h,hour_sin,hour_cos,dow_sin,dow_cos
0,1953986.0,0.028950,0.080053,-0.115968,-0.175979,-0.162894,-0.189824,-0.183605,-0.065961,56.795420,-2.449294e-15,1.000000,-1.203615e-02,0.024993
1,1953987.0,0.099730,0.160888,-0.056620,-0.107224,-0.102212,-0.144209,-0.111272,-0.061589,55.543889,2.083183e-02,0.999783,-1.203615e-02,0.024993
2,1953988.0,0.139315,0.222990,0.034206,-0.047842,-0.054766,-0.091555,-0.073619,-0.070867,57.067185,-9.799650e-15,1.000000,-1.203615e-02,0.024993
3,1953989.0,0.174408,0.274305,0.080942,0.006493,-0.011330,-0.027491,-0.053343,-0.047543,62.209395,2.083183e-02,0.999783,-1.203615e-02,0.024993
4,1953990.0,0.202848,0.312351,0.150501,0.054711,0.018790,-0.000740,-0.027370,-0.048842,64.349420,-2.939152e-15,1.000000,-1.203615e-02,0.024993
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
433995,1954007.0,-0.162531,-0.189577,-0.301255,-0.330513,-0.247720,-0.215433,-0.004593,-0.055683,63.555671,2.083183e-02,0.999783,-1.353931e-21,0.021975
433996,1954008.0,-0.099592,-0.101034,-0.231208,-0.294785,-0.237063,-0.236899,0.015918,-0.073452,62.380488,-2.204364e-15,1.000000,-1.353931e-21,0.021975
433997,1954009.0,-0.031774,-0.007298,-0.195814,-0.247002,-0.233560,-0.240582,-0.171279,-0.071179,58.476976,2.083183e-02,0.999783,-1.353931e-21,0.021975
433998,1953986.0,0.028950,0.080053,-0.115968,-0.175979,-0.162894,-0.189824,-0.183605,-0.065961,56.795420,-2.449294e-15,1.000000,-1.203615e-02,0.024993


In [40]:
rmse = 0;
crit = ['status_1', 'status_2', 'status_3', 'status_4', 'status_5', 'status_6', 'status_7', 'status_8']

for i in range(0, df_test_temp.shape[0]//32, 32):
    rmse += mean_squared_error(df_test_temp[crit].iloc[i:i+32], df_mean_expanded[crit].iloc[i:i+32])
rmse = math.sqrt(rmse/(df_test_temp.shape[0]//32))
rmse

0.3356258876187904

# Подготовка данных для RNN

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

In [4]:
df['hour_sin'] = np.sin(
    2 * np.pi * pd.to_datetime(df['timestamp']).dt.hour + pd.to_datetime(df['timestamp']).dt.minute/60 / 24
)
# df['hour_cos'] = np.cos(
#     2 * np.pi * pd.to_datetime(df['timestamp']).dt.hour + pd.to_datetime(df['timestamp']).dt.minute/60 / 24
# )

df['dow_sin'] = np.sin(2 * np.pi * df['timestamp'].dt.dayofweek / 7)
# df['dow_cos'] = np.cos(2 * np.pi * df['timestamp'].dt.dayofweek / 7)

In [5]:
df_train = pd.read_csv('df_train.csv')
df_test = pd.read_csv('df_test.csv')

In [ ]:
df_train = pd.DataFrame(columns=df.columns)
df_test = pd.DataFrame(columns=df.columns)

first_test_index = round(4342*0.9)

for group, data in df.groupby(["office_from_id", "route_id"]):
    df_train = pd.concat([df_train, data.reset_index(drop=True).iloc[:first_test_index]], ignore_index=True)
    df_test = pd.concat([df_test, data.reset_index(drop=True).iloc[first_test_index:]], ignore_index=True)
df_train

In [6]:
scalers = {}

for col in df.columns[3:11]:
    scaler = StandardScaler()
    df_train[col] = scaler.fit_transform(df_train[[col]])
    df_test[col] = scaler.transform(df_test[[col]])
    scalers[col] = scaler

In [7]:
class WarehouseDataset(Dataset):
    def __init__(self, data, seq_len=4):
        self.seq_len = seq_len
        self.status_cols = [f'status_{i}' for i in range(1, 9)]
        
        self.sequences = []
        self.offices = []
        self.routes = []
        self.hours = []
        self.dow = []
        self.targets = []
        
        for route_id, group in data.groupby('route_id'):
            office_id = group['office_from_id'].iloc[0]
            status_values = group[self.status_cols].values
            hours = group['hour_sin'].values
            dow = group['dow_sin'].values
            
            for i in range(len(status_values) - seq_len):
                self.sequences.append(status_values[i:i+seq_len])
                self.offices.append(office_id)
                self.routes.append(route_id)
                self.dow.append(dow[i:i+seq_len])
                self.hours.append(hours[i:i+seq_len])
                self.targets.append(status_values[i+seq_len])

    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        seq = torch.FloatTensor(self.sequences[idx])
        office = torch.LongTensor([self.offices[idx] - 1]) 
        route = torch.LongTensor([self.routes[idx]])
        hours = torch.FloatTensor(self.hours[idx])
        dow = torch.FloatTensor(self.dow[idx])
        target = torch.FloatTensor(self.targets[idx])
        
        return seq, office, route, hours, dow, target


In [8]:
class WarehouseLSTM(nn.Module):
    def __init__(self, input_size=8, hidden_size=128, num_layers=2, office_embedding_dim=8, route_embedding_dim=16):
        super().__init__()
        
        self.office_embedding = nn.Embedding(54, office_embedding_dim)
        self.route_embedding = nn.Embedding(1000, route_embedding_dim)
        
        self.dynamic_context_fc = nn.Linear(2, hidden_size)
        
        static_context_dim = office_embedding_dim + route_embedding_dim
        self.init_hidden_fc = nn.Linear(static_context_dim, hidden_size)
        self.init_cell_fc = nn.Linear(static_context_dim, hidden_size)
        
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=0.2)
        
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
#             nn.Dropout(0.2),
            nn.Linear(hidden_size // 2, input_size)
        )
    
    def forward(self, seq, office, route, hours, dow):
        batch_size, seq_len = seq.shape[:2]
        
        office_emb = self.office_embedding(office.squeeze())
        route_emb = self.route_embedding(route.squeeze())
        static_context = torch.cat([office_emb, route_emb], dim=1)
        
        h0 = self.init_hidden_fc(static_context).unsqueeze(0)
        c0 = self.init_cell_fc(static_context).unsqueeze(0)
        
        lstm_out, (h_n, c_n) = self.lstm(seq, (h0, c0))

        dynamic_context = torch.stack([hours, dow], dim=2)
        
        dynamic_effect = self.dynamic_context_fc(dynamic_context)
        
        combined = lstm_out + dynamic_effect
        
        last_lstm = combined[:, -1, :]
        
        output = self.fc(last_lstm)
        
        return output

In [9]:
def train_model(model, train_loader, val_loader, epochs=30, lr=0.001):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5)
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for seq, office, route,  hours, dow, target in train_loader:
            seq = seq.to(device)
            office = office.squeeze().to(device)
            route = route.squeeze().to(device)
            hours = hours.to(device)
            dow = dow.to(device)
            target = target.to(device)
            
            optimizer.zero_grad()
            output = model(seq, office, route, hours, dow)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
        
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for seq, office, route, hours, dow, target in val_loader:
                seq = seq.to(device)
                office = office.squeeze().to(device)
                route = route.squeeze().to(device)
                hours = hours.to(device)
                dow = dow.to(device)
                target = target.to(device)
                
                output = model(seq, office, route, hours, dow)
                val_loss += criterion(output, target).item()
        
        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        
        scheduler.step(avg_val_loss)
        
        print(f'Epoch {epoch}: Train Loss = {math.sqrt(avg_train_loss):.4f}, Val Loss = {math.sqrt(avg_val_loss):.4f}')
    
    return model

In [10]:
train_dataset = WarehouseDataset(df_train, seq_len=6)
val_dataset = WarehouseDataset(df_test, seq_len=6)

In [11]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [12]:
model = WarehouseLSTM(
        input_size=8,
        hidden_size=128,
        num_layers=1,
        office_embedding_dim=8,
        route_embedding_dim=16
    )

C:\Anaconda\envs\nntypical\Lib\site-packages\torch\nn\modules\rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn(


In [13]:
model = train_model(model, train_loader, val_loader, epochs=30)

C:\Users\Мишаня\AppData\Local\Temp\ipykernel_21120\4231263649.py:34: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\cb\pytorch_1000000000000\work\torch\csrc\utils\tensor_numpy.cpp:212.)
  hours = torch.FloatTensor(self.hours[idx])


Epoch 0: Train Loss = 0.4255, Val Loss = 0.4726


KeyboardInterrupt: 

In [ ]:
df_train.info()

In [ ]:
for col in ['route_id', 'office_from_id', 'dow_sin', 'hour_sin']:
    df_train[col] = pd.to_numeric(df_train[col], errors='coerce')
    df_test[col] = pd.to_numeric(df_test[col], errors='coerce')
